In [ ]:
from google.colab import drive
import os

def _drive_ready(mountpoint='/content/drive'):
    """진짜로 Drive 가 붙어 있는가.

    isdir('/content/drive/MyDrive') 만 보면 안 된다 — 마운트가 끊긴 자리에 같은 이름의
    폴더가 남으면 붙지도 않았는데 붙은 것처럼 보이고, 그 뒤 읽기·쓰기가 전부 조용히
    로컬 디스크로 새 나간다(한참 뒤 엉뚱한 데서 FileNotFoundError 로 터진다)."""
    return os.path.ismount(mountpoint) and any(
        os.path.isdir(os.path.join(mountpoint, n)) for n in ('MyDrive', 'My Drive')
    )


def _ensure_drive(mountpoint='/content/drive'):
    """이미 붙어 있으면 넘어가고, 아니면 붙인다. 마운트 자리에 찌꺼기가 남아
    mount 가 거부하는 경우("Mountpoint must not already contain files")까지 복구한다."""
    if _drive_ready(mountpoint):
        return
    try:
        drive.mount(mountpoint)
        return
    except ValueError:
        pass

    try:
        drive.flush_and_unmount()
    except Exception:
        pass
    if not os.path.ismount(mountpoint):
        # 마운트가 아니니 여기 있는 건 Drive 원본이 아니라 로컬 찌꺼기다. 지우지 말고
        # 옆으로 치워 둔다 — 이 상태에서 저장된 파일이 있었다면 살릴 수 있게.
        # (rm -rf 로 지우는 방법은 쓰지 않는다. 마운트가 살아 있었다면 원본이 날아간다.)
        stale, n = mountpoint + '_stale', 1
        while os.path.exists(stale):
            n += 1
            stale = f'{mountpoint}_stale{n}'
        os.rename(mountpoint, stale)
        print(f'마운트가 아닌 찌꺼기 폴더를 {stale} 로 치웠습니다.')
    drive.mount(mountpoint, force_remount=True)


_ensure_drive()

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import os
import re
import random
import unicodedata

from google.colab import drive
import os

def _drive_ready(mountpoint='/content/drive'):
    """진짜로 Drive 가 붙어 있는가.

    isdir('/content/drive/MyDrive') 만 보면 안 된다 — 마운트가 끊긴 자리에 같은 이름의
    폴더가 남으면 붙지도 않았는데 붙은 것처럼 보이고, 그 뒤 읽기·쓰기가 전부 조용히
    로컬 디스크로 새 나간다(한참 뒤 엉뚱한 데서 FileNotFoundError 로 터진다)."""
    return os.path.ismount(mountpoint) and any(
        os.path.isdir(os.path.join(mountpoint, n)) for n in ('MyDrive', 'My Drive')
    )


def _ensure_drive(mountpoint='/content/drive'):
    """이미 붙어 있으면 넘어가고, 아니면 붙인다. 마운트 자리에 찌꺼기가 남아
    mount 가 거부하는 경우("Mountpoint must not already contain files")까지 복구한다."""
    if _drive_ready(mountpoint):
        return
    try:
        drive.mount(mountpoint)
        return
    except ValueError:
        pass

    try:
        drive.flush_and_unmount()
    except Exception:
        pass
    if not os.path.ismount(mountpoint):
        # 마운트가 아니니 여기 있는 건 Drive 원본이 아니라 로컬 찌꺼기다. 지우지 말고
        # 옆으로 치워 둔다 — 이 상태에서 저장된 파일이 있었다면 살릴 수 있게.
        # (rm -rf 로 지우는 방법은 쓰지 않는다. 마운트가 살아 있었다면 원본이 날아간다.)
        stale, n = mountpoint + '_stale', 1
        while os.path.exists(stale):
            n += 1
            stale = f'{mountpoint}_stale{n}'
        os.rename(mountpoint, stale)
        print(f'마운트가 아닌 찌꺼기 폴더를 {stale} 로 치웠습니다.')
    drive.mount(mountpoint, force_remount=True)


_ensure_drive()

# ============================================================
# 0. PBP 파일명 패턴
# ============================================================
PBP_PATTERN = re.compile(r'^\d{4}_\d{2}_[a-z]+\.xlsx$')

BASE_URL = "http://www.korea-baseball.com"
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    )
}

# ============================================================
# 1. game_idx 수집 (PBP 폴더 기반)
# ============================================================
def get_game_idx_from_pbp_folders(base_dir, years=None):
    if years is None:
        years = range(2020, 2027)

    game_idx_by_year = {}

    for year in years:
        year_dir = os.path.join(base_dir, str(year))
        if not os.path.exists(year_dir):
            print(f"{year}년 폴더 없음, 스킵")
            continue

        ids = []

        for root, dirs, files in os.walk(year_dir):
            dirs[:] = [unicodedata.normalize('NFC', d) for d in dirs]

            for fname in files:
                fname = unicodedata.normalize('NFC', fname)
                if fname.startswith('~$'):
                    continue
                if not PBP_PATTERN.match(fname):
                    continue

                fpath = os.path.join(root, fname)
                try:
                    df = pd.read_excel(fpath, usecols=['game_id'])
                    game_ids = df['game_id'].dropna().astype(int).unique().tolist()
                    ids.extend(game_ids)
                except Exception as e:
                    print(f"  [{fname}] 읽기 실패: {e}")
                    continue

        ids = sorted(set(ids))
        game_idx_by_year[year] = ids
        print(f"{year}년: {len(ids)}개 game_idx 수집")

    return game_idx_by_year


# ============================================================
# 1-2. game_idx 수집 (특정 idx 직접 지정)
# ============================================================
def get_game_idx_manual(year_to_ids: dict):
    """
    특정 game_idx를 직접 지정할 때 사용.

    사용 예:
        game_idx_by_year = get_game_idx_manual({
            2021: [23509, 23515, 23520],
            2022: [25000, 25001],
        })

    실패 목록 재시도 예:
        game_idx_by_year = get_game_idx_manual({
            2021: failed[2021],
            2022: failed[2022],
        })
    """
    game_idx_by_year = {}
    for year, ids in year_to_ids.items():
        ids = sorted(set(int(i) for i in ids))
        game_idx_by_year[year] = ids
        print(f"{year}년: {len(ids)}개 game_idx (수동 지정)")
    return game_idx_by_year


# ============================================================
# 2. 타자 박스스코어 파싱
# ============================================================
def parse_batter_box(game_idx, session):
    url = f"{BASE_URL}/game/record_detail?game_idx={game_idx}"
    try:
        resp = session.get(url, headers=HEADERS, timeout=15)
        resp.raise_for_status()
    except Exception as e:
        print(f"    [요청 실패] game_idx={game_idx}: {e}")
        return []

    soup = BeautifulSoup(resp.text, "html.parser")
    sections = soup.find_all("div", class_="section_sumrec")
    records = []

    # 타자기록 섹션 순서 카운터 (0=초/원정, 1=말/홈)
    batter_section_idx = 0

    for section in sections:
        h4 = section.find("h4")
        if h4 is None:
            continue
        h4_text = h4.get_text(strip=True)
        if "타자기록" not in h4_text:
            continue

        team = h4_text.replace("타자기록", "").strip()
        topbot = "top" if batter_section_idx == 0 else "bot"
        batter_section_idx += 1

        # tb_body 우선, 없으면 summary="타자기록" 첫 번째 테이블 사용
        table = None
        tb_wr = section.find("div", class_="tb_wr")
        if tb_wr:
            tb_body = tb_wr.find("div", class_="tb_body")
            if tb_body:
                table = tb_body.find("table")

        if table is None:
            table = section.find("table", summary="타자기록")

        if table is None:
            continue

        rows = table.find("tbody").find_all("tr")

        for row in rows:
            if "sum" in row.get("class", []):
                continue

            ths = row.find_all("th")
            if len(ths) < 3:
                continue

            batting_order_text = ths[0].get_text(strip=True)
            try:
                batting_order = int(batting_order_text)
            except ValueError:
                continue

            pos_span = ths[1].find("span", class_="position")
            position = pos_span.get_text(separator="/", strip=True) if pos_span else ths[1].get_text(strip=True)

            player_raw = ths[2].get_text(strip=True)
            # 등번호가 비어 있는 선수가 있다("김동영()"). (\d+) 로는 매칭이 안 돼 이름에
            # 괄호가 그대로 남고, 그러면 season_full·선수 마스터와 이름이 안 붙는다
            # (2026 batter_box 22,034행 중 429행이 그랬다). 투수기록 쪽 _strip_name_no 와
            # 같은 규칙으로 빈 번호까지 떼어낸다.
            match = re.match(r'^(.*?)\((\d*)\D*$', player_raw)
            if match:
                player_name = match.group(1).strip()
                player_num = int(match.group(2)) if match.group(2) else None
            else:
                player_name = player_raw.strip()
                player_num = None

            tds = row.find_all("td")
            inning_cells = [td for td in tds if "fixed" not in td.get("class", [])]
            inning_texts = [td.get_text(strip=True) for td in inning_cells]
            is_pinch_hitter = any("대타" in t for t in inning_texts)

            fixed_tds = [td for td in tds if "fixed" in td.get("class", [])]
            numeric_fixed = []
            for td in fixed_tds:
                text = td.get_text(strip=True)
                try:
                    int(text)
                    numeric_fixed.append(text)
                except ValueError:
                    pass

            if len(numeric_fixed) < 4:
                continue

            try:
                ab  = int(numeric_fixed[0])
                h   = int(numeric_fixed[1])
                rbi = int(numeric_fixed[2])
                r   = int(numeric_fixed[3])
            except ValueError:
                continue

            records.append({
                "game_idx":        game_idx,
                "topbot":          topbot,
                "team":            team,
                "batting_order":   batting_order,
                "player_name":     player_name,
                "player_num":      player_num,
                "position":        position,
                "is_pinch_hitter": is_pinch_hitter,
                "AB":              ab,
                "H":               h,
                "RBI":             rbi,
                "R":               r,
            })

    return records


# ============================================================
# 3. 저장 함수
# ============================================================
def _save_batter_box(all_records, output_path):
    if not all_records:
        print("  [저장 스킵] 데이터 없음")
        return pd.DataFrame()

    df = pd.DataFrame(all_records)
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    df.to_excel(output_path, index=False)
    print(f"  [저장] {output_path} ({len(df)}행)")
    return df


# ============================================================
# 4. 연도별 크롤링 메인 함수
# ============================================================
def crawl_batter_box_by_year(game_idx_by_year, output_dir,
                              save_every=50, max_retry=3):
    os.makedirs(output_dir, exist_ok=True)
    total_failed = {}

    session = requests.Session()

    for year, game_ids in sorted(game_idx_by_year.items()):
        output_path = os.path.join(output_dir, f"batter_box_{year}.xlsx")
        all_records = []

        # 이미 수집된 game_idx 확인
        already_collected = set()
        if os.path.exists(output_path):
            try:
                existing_df = pd.read_excel(output_path, usecols=["game_idx"])
                already_collected = set(existing_df["game_idx"].dropna().astype(int).tolist())
                print(f"{year}년 기존 파일 발견: {len(already_collected)}경기 이미 수집됨")
                all_records = pd.read_excel(output_path).to_dict("records")
            except Exception as e:
                print(f"{year}년 기존 파일 읽기 실패 (새로 시작): {e}")

        remaining = [g for g in game_ids if g not in already_collected]

        print(f"\n{'='*50}")
        print(f"{year}년 타자 박스스코어 수집")
        print(f"전체 {len(game_ids)}경기 / 스킵 {len(already_collected)}경기 / 수집 {len(remaining)}경기")
        print(f"{'='*50}")

        if not remaining:
            print(f"{year}년 새로 수집할 경기 없음, 스킵")
            total_failed[year] = []
            continue

        failed = []

        for i, game_idx in enumerate(remaining):
            records = []
            elapsed = 0

            for attempt in range(1, max_retry + 1):
                start = time.time()
                records = parse_batter_box(game_idx, session)
                elapsed = time.time() - start

                if records:
                    print(f"  [{i+1}/{len(remaining)}] {game_idx} → {len(records)}명 ({elapsed:.1f}s)")
                    break
                else:
                    if attempt < max_retry:
                        retry_wait = random.uniform(9.0, 11.0)
                        print(f"  [{i+1}/{len(remaining)}] {game_idx} → 실패 ({elapsed:.1f}s) "
                              f"/ 재시도 {attempt}/{max_retry} ({retry_wait:.1f}s 후)")
                        time.sleep(retry_wait)
                    else:
                        print(f"  [{i+1}/{len(remaining)}] {game_idx} → 최종 실패 ({max_retry}회 시도)")
                        failed.append(game_idx)

            if records:
                all_records.extend(records)

            # 중간 저장
            if (i + 1) % save_every == 0:
                _save_batter_box(all_records, output_path)
                print(f"  >>> 중간 저장 완료 ({i+1}/{len(remaining)})")

            # 9~11초 대기 (응답시간 차감)
            target = random.uniform(9.0, 11.0)
            wait = max(0, target - elapsed)
            time.sleep(wait)

        # 최종 저장
        df = _save_batter_box(all_records, output_path)
        total_failed[year] = failed

        print(f"\n{year}년 완료: {len(df)}행 저장")
        print(f"실패 {len(failed)}개: {failed if failed else '없음'}")

    return total_failed


# ============================================================
# 실행 (둘 중 하나만 사용)
# ============================================================
BASE_DIR = '/content/drive/MyDrive/4-1/졸업작품 (고교야구 기록지)'
OUT_DIR  = os.path.join(BASE_DIR, 'batter_box')

# ── 방법 A: PBP 폴더 기반 전체 수집 ──────────────────────────
game_idx_by_year = get_game_idx_from_pbp_folders(BASE_DIR, years=[2022])

# ── 방법 B: 특정 game_idx 수동 지정 ──────────────────────────
# game_idx_by_year = get_game_idx_manual({
#     2021: [23515, 23713],
# })

# ── 방법 C: 실패 목록 재시도 ─────────────────────────────────
# game_idx_by_year = get_game_idx_manual({
#     2021: failed[2021],
#     2022: failed[2022],
# })

print("\n[수집 현황]")
for year, ids in sorted(game_idx_by_year.items()):
    print(f"  {year}년: {len(ids)}경기")

failed = crawl_batter_box_by_year(
    game_idx_by_year,
    output_dir=OUT_DIR,
    save_every=50,
    max_retry=3,
)

print("\n[최종 실패 목록]")
for year, ids in failed.items():
    if ids:
        print(f"  {year}년: {ids}")